# SMD — Exploratory Data Analysis

Quick look at the Server Machine Dataset for `machine-1-1`:
- Dimensions and data types
- Basic statistics
- Time-series plots of selected metrics
- Anomaly distribution

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from src.data import download_smd, load_machine

MACHINE = 'machine-1-1'
dataset_dir = download_smd('../data/raw')
train, test, labels = load_machine(dataset_dir, MACHINE)

print(f'Train shape: {train.shape}')
print(f'Test  shape: {test.shape}')
print(f'Labels — anomaly points: {labels.sum()} / {len(labels)} '
      f'({labels.mean()*100:.1f}%)')

In [ ]:
# Summary statistics for training data
import pandas as pd
df_train = pd.DataFrame(train, columns=[f'metric_{i}' for i in range(38)])
df_train.describe().T

In [ ]:
# Plot a few metrics from the test set with anomaly regions highlighted
fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
metrics_to_show = [0, 5, 10, 20]

for ax, m_idx in zip(axes, metrics_to_show):
    ax.plot(test[:, m_idx], linewidth=0.5)
    ax.fill_between(range(len(labels)), 0, labels * test[:, m_idx].max(),
                     alpha=0.2, color='red', label='anomaly')
    ax.set_ylabel(f'Metric {m_idx}')
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Time step')
fig.suptitle(f'{MACHINE} — Test set with anomaly regions', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Anomaly segment lengths
segments = []
in_seg = False
for i, l in enumerate(labels):
    if l == 1 and not in_seg:
        in_seg = True
        seg_start = i
    elif l == 0 and in_seg:
        in_seg = False
        segments.append(i - seg_start)
if in_seg:
    segments.append(len(labels) - seg_start)

print(f'Number of anomaly segments: {len(segments)}')
print(f'Segment lengths — min: {min(segments)}, max: {max(segments)}, '
      f'mean: {np.mean(segments):.1f}')

plt.figure(figsize=(8, 3))
plt.hist(segments, bins=20, edgecolor='black')
plt.xlabel('Segment length')
plt.ylabel('Count')
plt.title('Distribution of anomaly segment lengths')
plt.tight_layout()
plt.show()